<a href="https://colab.research.google.com/github/FYasmin12/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import os, duckdb
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
SRC = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print(con.sql(f"DESCRIBE SELECT * FROM {SRC}").df()[["column_name", "column_type"]])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. One row means: in my feature frame, one row = one page (content_hash_id) of one
   client, summarised over 1-14 March 2026. The raw table has one row per page per day.
2. Table: fact_content_daily_performance, only the month=2026-03 partition. I do not
   use the _sample table: it is June 2026, my sealed test month.
3. Time window: features from 1-14 March (decision moment: 15 March). Outcome window:
   15-28 March. Both are 14 days so the comparison is fair.
4. What I rank (proxy): is_declining = 1 if a page's impressions in 15-28 March are
   below 80% of its impressions in 1-14 March. This is a defined rule on a later,
   observed window, so it is still a proxy, not proof of decline. Output: pages ranked
   by decline risk for a reviewer.
5. Deliberately excluded: everything dated 15 March or later as a feature, and the
   IDs as features (IDs are for grouping and splitting only).

In [5]:
import datetime as dt

EARLY = (dt.date(2026, 3, 1),  dt.date(2026, 3, 14))
LATE  = (dt.date(2026, 3, 15), dt.date(2026, 3, 28))
DECLINE_THRESHOLD = 0.8

for name, (a, b) in [("early (features)", EARLY), ("late (outcome)", LATE)]:
    print(f"{name:18} {a} -> {b}  ({(b - a).days + 1} days)")
print("Decline rule: imp_late < ", DECLINE_THRESHOLD, "* imp_early")

early (features)   2026-03-01 -> 2026-03-14  (14 days)
late (outcome)     2026-03-15 -> 2026-03-28  (14 days)
Decline rule: imp_late <  0.8 * imp_early


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: imp_early, ctr_early, pos_early, active_days_early, momentum_wk2_vs_wk1
Label / proxy: imp_late and is_declining
Context: client_hash_id, content_hash_id (grouping and splitting only)
Excluded: all GA4 columns (the availability flag has three values and I do not need
engagement for this slice), anything after 14 March, the _sample table

In [6]:
FIELDS = {
    "feature": ["imp_early", "ctr_early", "pos_early", "active_days_early", "momentum_wk2_vs_wk1"],
    "label_or_proxy": ["imp_late", "is_declining"],
    "context": ["client_hash_id", "content_hash_id"],
    "excluded": {
        "ga4_* columns": "availability flag is TRUE / FALSE / NULL and I do not need engagement here",
        "anything dated after 14 March": "future information, would be leakage",
        "_sample table": "it is June 2026, my sealed test month",
    },
}

for bucket, items in FIELDS.items():
    print(f"\n{bucket.upper()}")
    if isinstance(items, dict):
        for k, why in items.items():
            print(f"  - {k}: {why}")
    else:
        print("  -", ", ".join(items))


FEATURE
  - imp_early, ctr_early, pos_early, active_days_early, momentum_wk2_vs_wk1

LABEL_OR_PROXY
  - imp_late, is_declining

CONTEXT
  - client_hash_id, content_hash_id

EXCLUDED
  - ga4_* columns: availability flag is TRUE / FALSE / NULL and I do not need engagement here
  - anything dated after 14 March: future information, would be leakage
  - _sample table: it is June 2026, my sealed test month


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1: grain

In [7]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {SRC} GROUP BY ALL HAVING c > 1 LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


Query 2: row count and date span:

In [8]:
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages
    FROM {SRC}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_day,last_day,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


Query 3: availability

In [9]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS survive_is_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS is_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS is_null
    FROM {SRC}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,survive_is_true,is_false,is_null
0,9841378,413966,6408671,3018741


In [10]:
D = "report_date BETWEEN DATE '{a}' AND DATE '{b}'"
early = D.format(a="2026-03-01", b="2026-03-14")
wk1   = D.format(a="2026-03-01", b="2026-03-07")
wk2   = D.format(a="2026-03-08", b="2026-03-14")
late  = D.format(a="2026-03-15", b="2026-03-28")

feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) FILTER (WHERE {early}) AS imp_early,
      SUM(gsc_clicks)      FILTER (WHERE {early}) AS clicks_early,
      AVG(gsc_avg_position) FILTER (WHERE {early} AND gsc_impressions > 0) AS pos_early,
      COUNT(*) FILTER (WHERE {early} AND gsc_impressions > 0) AS active_days_early,
      SUM(gsc_impressions) FILTER (WHERE {wk1}) AS imp_wk1,
      SUM(gsc_impressions) FILTER (WHERE {wk2}) AS imp_wk2,
      SUM(gsc_impressions) FILTER (WHERE {late}) AS imp_late
    FROM {SRC} GROUP BY 1, 2 HAVING imp_early >= 100
""").df()

feat["ctr_early"] = feat["clicks_early"] / feat["imp_early"]
feat["momentum_wk2_vs_wk1"] = feat["imp_wk2"] / feat["imp_wk1"].replace(0, float("nan"))
feat["is_declining"] = (feat["imp_late"].fillna(0) < 0.8 * feat["imp_early"]).astype(int)

FEATURES = ["imp_early", "ctr_early", "pos_early", "active_days_early", "momentum_wk2_vs_wk1"]
model_df = feat.dropna(subset=FEATURES).copy()
print(len(model_df), "pages | declining share:", round(model_df["is_declining"].mean(), 3))
model_df[FEATURES + ["is_declining"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

73882 pages | declining share: 0.345


,imp_early,ctr_early,pos_early,active_days_early,momentum_wk2_vs_wk1,is_declining
0,179.0,0.011173,4.105758,12,1.081395,0
1,443.0,0.002257,4.399802,12,1.150485,1
2,710.0,0.001408,1.883335,13,0.783920,0
3,220.0,0.000000,4.836081,12,1.619048,0
4,208.0,0.009615,3.835395,12,2.151515,0


All five use only 1-14 March, so they are all knowable on 15 March, the decision moment:
- imp_early: total impressions over 1-14 March, fully observed by 14 March.
- ctr_early: clicks / impressions over the same days.
- pos_early: average position on days the page had impressions.
- active_days_early: how many of those 14 days the page was shown.
- momentum_wk2_vs_wk1: second-week vs first-week impressions, both before 15 March.

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(frame, cols):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
                  .split(frame, groups=frame["client_hash_id"]))
    m = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    m.fit(frame.iloc[tr][cols], frame.iloc[tr]["is_declining"])
    return roc_auc_score(frame.iloc[te]["is_declining"], m.predict_proba(frame.iloc[te][cols])[:, 1])

honest = quick_score(model_df, FEATURES)

model_df["late_vs_early"] = model_df["imp_late"] / model_df["imp_early"]
leaky = quick_score(model_df, FEATURES + ["late_vs_early"])

model_df = model_df.drop(columns=["late_vs_early"])
print(f"honest AUC: {honest:.3f}")
print(f"leaky  AUC: {leaky:.3f}  (jumps toward 1.0 because the column contains the answer)")

honest AUC: 0.692
leaky  AUC: 1.000  (jumps toward 1.0 because the column contains the answer)


I added late_vs_early, built from the outcome window (15-28 March), and the score
jumped toward perfect. That is leakage: the column is the label in disguise. I
deleted it and keep the honest number above.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: this slice is one mid-panel month, with a rule-based label (a 20% drop
between two 14-day windows), so ordinary noise and weekly patterns can flip a page's
label. I also keep only pages with at least 100 early impressions, so my results say
nothing about low-traffic pages. The panel is unbalanced (clients start tracking at
different dates), so some clients have partial windows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.